# 🌸 Manga Translator Studio - Google Colab AI GPU Backend (v3.0 All-In-One 1-Click)
### 📌 Hướng dẫn 1 Click:
1. Chọn Menu **Runtime > Change runtime type > T4 GPU**.
2. (Tùy chọn) Nhập Authtoken ngrok của bạn vào ô `NGROK_AUTH_TOKEN` ở Bước 1 bên dưới.
3. Nhấn vào nút Play **▶ (Run Cell)** ở Bước 1.
4. Copy đường dẫn `https://xxxx.ngrok-free.app` hiển thị ở kết quả và dán vào ô **Kết nối Colab GPU** trên Web App!

In [ ]:
# ==========================================================================
# BƯỚC 1: KHỞI CHẠY TẤT CẢ TRONG 1 CLICK (ALL-IN-ONE GPU AI SERVER)
# ==========================================================================
NGROK_AUTH_TOKEN = "3JDrqEhhN12GBwXRsmReBWBeRYI_3RYgo1BLNrm7AmUmghaf5" # <-- DÁN NGROK TOKEN CỦA BẠN VÀO ĐÂY NẾU CÓ

import os, sys, time, io, base64, urllib.request, threading, subprocess, json, re
import numpy as np
from PIL import Image

print("⏳ [1/4] Đang cài đặt thư viện cần thiết...")
os.system("pip install -q fastapi uvicorn pyngrok pydantic pillow deep-translator google-generativeai nest-asyncio transformers sentencepiece fugashi unidic-lite manga-ocr")
os.system("npm install -g localtunnel > /dev/null 2>&1")

import torch
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional, Dict, Any
from deep_translator import GoogleTranslator
from pyngrok import ngrok

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 [2/4] Thiết bị tính toán GPU: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# 1. Nạp mô hình LaMa Inpainting (TorchScript Native)
class StandaloneLama:
    def __init__(self, dev="cuda"):
        self.dev = torch.device(dev if torch.cuda.is_available() else "cpu")
        model_path = "big-lama.pt"
        if not os.path.exists(model_path) or os.path.getsize(model_path) < 10000000:
            print("⏳ Đang tải mô hình LaMa (big-lama.pt ~196MB)... Vui lòng đợi trong giây lát...")
            url = "https://github.com/Sanster/models/releases/download/add_big_lama/big-lama.pt"
            urllib.request.urlretrieve(url, model_path)
            print("✅ Đã tải mô hình LaMa thành công!")
        
        print("⏳ Đang nạp mô hình LaMa vào GPU...")
        self.model = torch.jit.load(model_path, map_location=self.dev)
        self.model.eval()
        print("✅ LaMa Inpainting sẵn sàng!")

    def __call__(self, img: Image.Image, mask: Image.Image) -> Image.Image:
        orig_w, orig_h = img.size
        mod_w = max(8, (orig_w // 8) * 8)
        mod_h = max(8, (orig_h // 8) * 8)
        img_resized = img.resize((mod_w, mod_h), Image.Resampling.BILINEAR)
        mask_resized = mask.resize((mod_w, mod_h), Image.Resampling.NEAREST).convert("L")
        img_np = np.array(img_resized).astype(np.float32) / 255.0
        mask_np = (np.array(mask_resized).astype(np.float32) / 255.0 > 0.5).astype(np.float32)
        img_t = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).to(self.dev)
        mask_t = torch.from_numpy(mask_np).unsqueeze(0).unsqueeze(0).to(self.dev)
        with torch.no_grad():
            res_t = self.model(img_t, mask_t)
        res_np = res_t[0].permute(1, 2, 0).detach().cpu().numpy()
        res_np = np.clip(res_np * 255.0, 0, 255).astype(np.uint8)
        return Image.fromarray(res_np).resize((orig_w, orig_h), Image.Resampling.BILINEAR)

lama = StandaloneLama(dev=device)

# 2. Nạp Manga-OCR cho tiếng Nhật
mocr = None
try:
    print("⏳ [3/4] Đang nạp Manga-OCR (Chế độ Uncensored)... Grounding...")
    from manga_ocr import MangaOcr
    mocr = MangaOcr(force_cpu=True)
    print("✅ Manga-OCR sẵn sàng!")
except Exception as me:
    print(f"⚠️ Manga-OCR note: {me}")

# 3. Khởi tạo FastAPI Server
app = FastAPI(title="Manga Translator Studio AI Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

def b64_to_pil(b64_str: str) -> Image.Image:
    if "," in b64_str: b64_str = b64_str.split(",")[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str))).convert("RGB")

def pil_to_b64(pil_img: Image.Image, fmt="PNG") -> str:
    buf = io.BytesIO()
    pil_img.save(buf, format=fmt)
    return f"data:image/{fmt.lower()};base64," + base64.b64encode(buf.getvalue()).decode("utf-8")

class InpaintReq(BaseModel):
    imageBase64: str
    maskBase64: str
class OcrReq(BaseModel):
    imageBase64: str
    boxes: List[Dict[str, Any]]
class TransReq(BaseModel):
    texts: List[str]
    sourceLang: Optional[str] = "auto"
    targetLang: Optional[str] = "vi"
    engine: Optional[str] = "google"
    apiKey: Optional[str] = None

@app.get("/health")
def health():
    return {"status": "online", "device": device, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", "lama_ready": True, "mocr_ready": mocr is not None}

@app.post("/api/inpaint")
async def do_inpaint(req: InpaintReq):
    try:
        img = b64_to_pil(req.imageBase64)
        mask = b64_to_pil(req.maskBase64).convert("L")
        res = lama(img, mask)
        return {"success": True, "cleanedImageBase64": pil_to_b64(res)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/ocr")
async def do_ocr(req: OcrReq):
    try:
        img = b64_to_pil(req.imageBase64)
        results = []
        for box in req.boxes:
            x, y = max(0, int(box.get("x", 0))), max(0, int(box.get("y", 0)))
            w, h = max(10, int(box.get("width", 50))), max(10, int(box.get("height", 50)))
            crop = img.crop((x, y, min(img.width, x + w), min(img.height, y + h)))
            text = mocr(crop) if mocr is not None else "(OCR Text)"
            results.append({"box": box, "text": text})
        return {"success": True, "results": results}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/translate")
async def do_translate(req: TransReq):
    try:
        if req.engine == "gemini" and req.apiKey:
            import google.generativeai as genai
            genai.configure(api_key=req.apiKey)
            model = genai.GenerativeModel("gemini-2.0-flash")
            prompt = f"Dịch các câu manga sau sang tiếng Việt:\n" + "\n".join(f"{i+1}. {t}" for i, t in enumerate(req.texts))
            res = model.generate_content(prompt)
            m = re.search(r'\[.*\]', res.text, re.DOTALL)
            out = json.loads(m.group(0)) if m else [res.text.strip()]
        else:
            translator = GoogleTranslator(source=req.sourceLang or "auto", target=req.targetLang or "vi")
            out = [translator.translate(t) if t and t.strip() else "" for t in req.texts]
        return {"success": True, "translated": out}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 4. Giải phóng cổng 8000 & Bật Server Uvicorn trong luồng Daemon
os.system("fuser -k 8000/tcp > /dev/null 2>&1")
time.sleep(1)
import uvicorn
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="error")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
time.sleep(2)

# 5. Mở Cổng Public (Auto Ngrok / Cloudflare Tunnel / LocalTunnel)
public_url = None

if NGROK_AUTH_TOKEN and len(NGROK_AUTH_TOKEN.strip()) > 10:
    try:
        for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
        ngrok.kill()
        ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())
        tunnel = ngrok.connect(8000)
        public_url = tunnel.public_url
    except Exception as ne:
        print(f"⚠️ Ngrok note: {ne}")

if not public_url:
    print("⏳ Đang tạo cổng kết nối Cloudflare Tunnel miễn phí (Không cần authtoken)...")
    try:
        os.system("curl -L -s https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /tmp/cloudflared")
        os.system("chmod +x /tmp/cloudflared")
        cf_log = open("/tmp/cf.log", "w")
        subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=cf_log, stderr=cf_log)
        time.sleep(3)
        for _ in range(12):
            if os.path.exists("/tmp/cf.log"):
                content = open("/tmp/cf.log").read()
                m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
                if m:
                    public_url = m.group(0)
                    break
            time.sleep(1)
    except Exception as ce:
        print(f"⚠️ Cloudflare note: {ce}")

if not public_url:
    print("⏳ Đang chuyển sang cổng LocalTunnel...")
    try:
        lt_log = open("/tmp/lt.log", "w")
        subprocess.Popen(["npx", "localtunnel", "--port", "8000"], stdout=lt_log, stderr=lt_log)
        time.sleep(4)
        if os.path.exists("/tmp/lt.log"):
            m = re.search(r"https://[a-zA-Z0-9-]+\.loca\.lt", open("/tmp/lt.log").read())
            if m: public_url = m.group(0)
    except Exception as le:
        print(f"⚠️ LocalTunnel note: {le}")
print("="*65)
print(f"🎉 [4/4] SERVER COLAB AI ĐÃ SẴN SÀNG KHỞI CHẠY!")
if public_url:
    print(f"🔗 COPY URL NÀY DÁN VÀO WEB APP: {public_url}")
else:
    print("🔗 Mở cổng local: http://localhost:8000")
print("="*65)

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Server đã dừng.")

⏳ [1/4] Đang cài đặt thư viện cần thiết...
🔥 [2/4] Thiết bị tính toán GPU: cuda (Tesla T4)
⏳ Đang nạp mô hình LaMa vào GPU...
✅ LaMa Inpainting sẵn sàng!
⏳ [3/4] Đang nạp Manga-OCR (Chế độ Uncensored)... Grounding...


2026-10-01 13:40:00.777 | INFO     | manga_ocr.ocr:__init__:16 - Loading OCR model from kha-white/manga-ocr-base
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/264 [00:00<?, ?it/s]

2026-10-01 13:40:12.816 | INFO     | manga_ocr.ocr:__init__:30 - Using CPU
2026-10-01 13:40:14.075 | INFO     | manga_ocr.ocr:__init__:37 - OCR ready


✅ Manga-OCR sẵn sàng!


ERROR:pyngrok.process.ngrok:t=2026-10-01T13:40:17+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-10-01T13:40:17+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
CRITICAL:pyngrok.process.ngrok:t=2026-10-01T13:40:17+0000 lvl=crit msg="command failed" err="authentication failed: This ngrok session is not authenticated. ngrok requi

⚠️ Ngrok note: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.
⏳ Đang tạo cổng kết nối Cloudflare Tunnel miễn phí (Không cần authtoken)...
🎉 [4/4] SERVER COLAB AI ĐÃ SẴN SÀNG KHỞI CHẠY!
🔗 COPY URL NÀY DÁN VÀO WEB APP: https://charlie-predicted-african-quarters.trycloudflare.com
